In [4]:
# Dependency Installation
!pip install -q ultralytics opencv-python-headless ffmpeg-python

In [5]:
# Import required libraries for video processing and YOLO model usage
from ultralytics import YOLO
from IPython.display import Video, display
import subprocess
import glob
import os
import shutil

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [6]:
# General settings

# Input video path
VIDEO_PATH = "/kaggle/input/fighter-aircraft-video-dataset/F-16.mp4"

# Working directories
WORK_DIR = "/kaggle/working"
PROJECT_DIR = os.path.join(WORK_DIR, "runs")  # where Ultralytics stores results
TRACK_NAME = "fighter_track"                  # fixed experiment name

print("Input video:", VIDEO_PATH)
print("Working directory:", WORK_DIR)

Input video: /kaggle/input/fighter-aircraft-video-dataset/F-16.mp4
Working directory: /kaggle/working


In [7]:
# Clean previous results
# This ensures there will always be a single AVI file and a single result.mp4

if os.path.exists(PROJECT_DIR):
    print("Deleting previous results in:", PROJECT_DIR)
    shutil.rmtree(PROJECT_DIR)
else:
    print("There were no previous results to delete")

There were no previous results to delete


In [8]:
# YOLO Tracking (fighter jets only)

model = YOLO("yolov8n.pt")

print("Processing video and tracking ONLY aircraft (fighter jets)...")

results = model.track(
    source=VIDEO_PATH,
    save=True,
    persist=True,
    stream=True,
    conf=0.20,
    classes=[4],           # class 4 = airplane in COCO
    imgsz=1280,
    tracker="botsort.yaml",
    project=PROJECT_DIR,   # always in /kaggle/working/runs
    name=TRACK_NAME,       # fixed experiment name
    exist_ok=True          # overwrite if it already exists
)

# Consume the generator to ensure processing completes
for _ in results:
    pass

print("Processing finished")

Processing video and tracking ONLY aircraft (fighter jets)...
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.11.13 environment at: /usr
Resolved 2 packages in 227ms
Prepared 1 package in 79ms
Installed 1 package in 5ms
 + lap==0.5.12

requirements: AutoUpdate success ✅ 0.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


video 1/1 (frame 1/289) /kaggle/input/fighter-aircraft-video-dataset/F-16.mp4: 1280x736 (no detections), 59.3ms
video 1/1 (frame 2/289) /kaggle/input/fighter-aircraft-video-dataset/F-16.mp4: 1280x736 3 airplanes, 13.3ms
video 1/1 (frame 3/289) /kaggle/input/fighter-aircraft-video-dataset/F-16.mp4: 1280x736 (no detections), 13.3ms
video 1/1 (frame 4/289) /kaggle/input/fighter-aircraft-video-dataset/F-16.mp4: 1280x736 1 airplane, 13.3ms
video 1/1 (frame 5/289) /kaggle/input/fighter-aircraft-video-dataset/F-16.mp4: 1280x736 1 airplane, 13.3ms
video 1/1 (frame 6/289) /kaggle/in

In [9]:
# Locate the single generated .avi file

print("Searching for generated video (.avi) inside:", PROJECT_DIR)

avi_files = glob.glob(os.path.join(PROJECT_DIR, "**", "*.avi"), recursive=True)

if not avi_files:
    raise FileNotFoundError("No AVI file generated by YOLO was found")

AVI_PATH = avi_files[0]  # since we cleaned beforehand, there will only be one
print("AVI video found:", AVI_PATH)

Searching for generated video (.avi) inside: /kaggle/working/runs
AVI video found: /kaggle/working/runs/fighter_track/F-16.avi


In [10]:
# Always convert to result.mp4 (overwritten on each run)

OUTPUT_MP4 = os.path.join(WORK_DIR, "result.mp4")

print("Converting to MP4 (result.mp4)...")

subprocess.run([
    "ffmpeg", "-y",          # -y = overwrite without prompting
    "-i", AVI_PATH,
    "-vcodec", "libx264",
    "-acodec", "aac",
    OUTPUT_MP4
], check=True)

print("MP4 generated:", OUTPUT_MP4)

Converting to MP4 (result.mp4)...


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

MP4 generated: /kaggle/working/result.mp4


frame=  289 fps= 34 q=-1.0 Lsize=    2872kB time=00:00:09.86 bitrate=2385.7kbits/s speed=1.18x    
video:2868kB audio:0kB subtitle:0kB other streams:0kB global headers:0kB muxing overhead: 0.133125%
[libx264 @ 0x5b4ec67e16c0] frame I:2     Avg QP:20.94  size: 59676
[libx264 @ 0x5b4ec67e16c0] frame P:126   Avg QP:22.46  size: 15654
[libx264 @ 0x5b4ec67e16c0] frame B:161   Avg QP:24.21  size:  5246
[libx264 @ 0x5b4ec67e16c0] consecutive B-frames: 18.7% 15.2% 17.6% 48.4%
[libx264 @ 0x5b4ec67e16c0] mb I  I16..4: 31.7% 63.1%  5.2%
[libx264 @ 0x5b4ec67e16c0] mb P  I16..4: 13.7% 36.0%  1.1%  P16..4: 24.9%  6.3%  3.2%  0.0%  0.0%    skip:14.9%
[libx264 @ 0x5b4ec67e16c0] mb B  I16..4:  3.2%  7.7%  0.1%  B16..8: 38.3%  5.4%  0.6%  direct: 1.3%  skip:43.4%  L0:50.2% L1:42.8% BI: 7.1%
[libx264 @ 0x5b4ec67e16c0] 8x8 transform intra:70.5% inter:76.3%
[libx264 @ 0x5b4ec67e16c0] coded y,uvDC,uvAC intra: 29.7% 35.7% 1.2% inter: 9.4% 10.5% 0.3%
[libx264 @ 0x5b4ec67e16c0] i16 v,h,dc,p: 20% 46% 13% 22%
[l

In [11]:
# Display the final video

print("Displaying final video: result.mp4")
display(Video(OUTPUT_MP4, embed=True))

Displaying final video: result.mp4
